# Gene Embedding Extraction from Single-Cell Foundation Models

This notebook extracts gene embeddings from multiple single-cell foundation models for downstream analysis. The included models are:

- **xTrimoGene**: Foundation model for single-cell genomics
- **Geneformer**: Transformer pretrained on single-cell transcriptomes  
- **LangCell**: Combines cell biology knowledge with language modeling
- **scCello**: Uses contrastive learning for single-cell analysis
- **scGPT**: Generative transformer pretrained for single-cell analysis
- **UCE**: Universal cell embeddings using ESM2 protein embeddings

The extracted embeddings will be saved in csv format.

## 1. Import Libraries and Set Paths

First, we import all the necessary libraries to load the models and process gene embeddings. We also set up the paths where the pretrained models are located.


In [ ]:
import os 
import sys
# Agregar rutas al path para importar módulos específicos
sys.path.insert(0, "./sc_foundation_evals")
sys.path.insert(0, "./xTrimoGene/model")

import pandas as pd
import torch
from tqdm import tqdm
import numpy as np
import pickle
from collections import defaultdict

# Importar módulos específicos de cada modelo
from xTrimoGene.model.load import load_model_frommmf
from sc_foundation_evals import scgpt_forward
from transformers import BertForMaskedLM, BertModel
from sc_foundation_evals.sccello.src.model_prototype_contrastive import PrototypeContrastiveModel

print("✅ Libraries imported successfully.")

## 2. Load Gene Data

We load the CSV file containing the mapping between gene identifiers (GeneID) and gene symbols (Symbol). This file will be used to standardize gene identifiers across all models.


In [ ]:
# Load gene identifier mapping
gene_set_df = pd.read_csv("./FRoGS/data/gene_id2symbol.csv")
print(f"📊 Loaded gene mapping with {len(gene_set_df)} entries")

# Define directory where pretrained models are located
parent_model_dir = "/mnt/nvme/extra_data/wujialu/scFM-Bench/data/weights"
print(f"\n📁 Model directory: {parent_model_dir}")


## 3. Extract Gene Embeddings with xTrimoGene

**xTrimoGene** is a foundation model for single-cell genomics.

### Procedure:
1. Load the gene list specific to xTrimoGene
2. Load the pretrained model from the checkpoint
3. Extract the positional embeddings representing the genes
4. Process and save the embeddings in CSV format


In [ ]:
print("🧬 Extracting embeddings from xTrimoGene...")

# Load the gene list specific to xTrimoGene
gene_list_file = f"{parent_model_dir}/scFoundation/OS_scRNA_gene_index.19264.tsv"
gene_list_df = pd.read_csv(gene_list_file, header=0, delimiter='\t')
gene_list = list(gene_list_df['gene_name'])

print(f"📋 Loaded {len(gene_list)} genes from xTrimoGene")

# Load the pretrained model from the checkpoint
ckpt_path = f"{parent_model_dir}/scFoundation/models.ckpt"
key = "cell"
pretrainmodel, pretrainconfig = load_model_frommmf(ckpt_path, key)


In [ ]:
#Extract the positional embeddings representing the genes
token_emb = pretrainmodel.pos_emb.weight
print(f"🔍 Shape of extracted embeddings: {token_emb.shape}")
print(f"Taking first 19264 embeddings for genes")

# Process and save xTrimoGene embeddings
gene_emb_df = pd.DataFrame(token_emb.detach().numpy()[:19264,:])
gene_emb_df["Symbol"] = gene_list

# Merge with reference gene set
gene_emb_df = pd.merge(left=gene_set_df, right=gene_emb_df, on="Symbol", how="inner")
gene_emb_df.set_index("GeneID", inplace=True)
gene_emb_df.drop(["Symbol"], axis=1, inplace=True)

# Save to CSV
output_file = f"./FRoGS/gene_embs/gene_vec_xtrimogene_{gene_emb_df.shape[1]}.csv"
gene_emb_df.to_csv(output_file, header=None)

print(f"✅ xTrimoGene embeddings saved: {gene_emb_df.shape}")
print(f"📁 File: {output_file}")
print(f"🧬 Genes processed: {len(gene_emb_df)} of {len(gene_list)} originals")


## 4. Extract Gene Embeddings with Geneformer

**Geneformer** is a transformer pretrained on single-cell transcriptomes. The embeddings are extracted from the word embedding layer of the BERT model.

### Procedure:
1. Load the pretrained Geneformer model
2. Load token dictionaries and gene name mapping
3. Extract embeddings from the word embedding layer
4. Map tokens to gene symbols and process
5. Save embeddings in CSV format


In [ ]:
print("🧬 Extracting embeddings from Geneformer...")

# Load the pretrained Geneformer model
saved_model_path = f"{parent_model_dir}/Geneformer/default/12L"
model = BertForMaskedLM.from_pretrained(saved_model_path,
                                        output_attentions=False,
                                        output_hidden_states=True)

print(f"🤖 Geneformer model loaded from: {saved_model_path}")


In [ ]:
# Load token dictionaries and gene name mappings for Geneformer
dict_paths = f"{parent_model_dir}/Geneformer/dicts"

# Token dictionary
token_dictionary_path = os.path.join(dict_paths, "token_dictionary.pkl")
with open(token_dictionary_path, "rb") as f:
    vocab = pickle.load(f)

# Gene name-ID mapping dictionary
gene_name_id_path = os.path.join(dict_paths, "gene_name_id_dict.pkl")
with open(gene_name_id_path, "rb") as f:
    gene_name_id = pickle.load(f)

# Create reverse ID-name mapping
gene_id_name = {v: k for k, v in gene_name_id.items()}

print(f"📚 Vocabulario cargado: {len(vocab)} tokens")
print(f"🧬 Mapeo de genes cargado: {len(gene_name_id)} genes")
print(f"🔍 Token PAD ID: {vocab.get('<pad>')}")


In [ ]:
# Extract embeddings from the word embedding layer
token_emb = model.state_dict()['bert.embeddings.word_embeddings.weight']
print(f"🔍 Shape of Geneformer embeddings: {token_emb.shape}")

# Create DataFrame with embeddings
gene_emb_df = pd.DataFrame(token_emb.numpy())
gene_emb_df["ENSG_ID"] = vocab.keys()
gene_emb_df["Symbol"] = gene_emb_df.apply(lambda x: gene_id_name.get(x["ENSG_ID"], None), axis=1)

# Filter valid genes and merge with reference set
gene_emb_df = pd.merge(left=gene_set_df, right=gene_emb_df, on="Symbol", how="inner")
gene_emb_df.set_index("GeneID", inplace=True)
gene_emb_df.drop(["Symbol","ENSG_ID"], axis=1, inplace=True)

# Save to CSV
output_file = f"./FRoGS/gene_embs/gene_vec_geneformer_{gene_emb_df.shape[1]}.csv"
gene_emb_df.to_csv(output_file, header=None)

print(f"✅ Geneformer embeddings saved: {gene_emb_df.shape}")
print(f"📁 File: {output_file}")

## 5. Extract Gene Embeddings with LangCell

**LangCell** combines cell biology knowledge with language modeling. It uses the same vocabulary as Geneformer but with a specific BERT model.

### Procedure:
1. Load the pretrained LangCell model
2. Extract embeddings from the word embedding layer
3. Remove the CLS (classification) token, which does not represent a gene
4. Use the same vocabulary mapping as Geneformer
5. Process and save the embeddings

In [ ]:
print("🧬 Extracting embeddings from LangCell...")

# Load pretrained LangCell model
model = BertModel.from_pretrained(f"{parent_model_dir}/LangCell/cell_bert")
token_emb = model.state_dict()['embeddings.word_embeddings.weight']

print(f"🤖 LangCell model loaded")
print(f"🔍 Embedding shape: {token_emb.shape}")
print(f"⚠️  Removing last token (CLS) - new shape: {token_emb[:-1,:].shape}")


In [ ]:
# Process LangCell embeddings (removing CLS token)
gene_emb_df = pd.DataFrame(token_emb.numpy()[:-1, :])  # remove cls token
gene_emb_df["ENSG_ID"] = vocab.keys()
gene_emb_df["Symbol"] = gene_emb_df.apply(lambda x: gene_id_name.get(x["ENSG_ID"], None), axis=1)

# Merge with reference gene set
gene_emb_df = pd.merge(left=gene_set_df, right=gene_emb_df, on="Symbol", how="inner")
gene_emb_df.set_index("GeneID", inplace=True)
gene_emb_df.drop(["Symbol", "ENSG_ID"], axis=1, inplace=True)

# Save to CSV
output_file = f"./FRoGS/gene_embs/gene_vec_langcell_{gene_emb_df.shape[1]}.csv"
gene_emb_df.to_csv(output_file, header=None)

print(f"✅ LangCell embeddings saved: {gene_emb_df.shape}")
print(f"📁 File: {output_file}")

## 6. Extract Gene Embeddings with scCello

**scCello** uses contrastive learning for single-cell analysis. It employs a prototype contrastive model to generate gene embeddings.

### Procedure:
1. Load the pretrained scCello model
2. Extract embeddings from the word embedding layer
3. Remove the CLS token (as in LangCell)
4. Use the same vocabulary as previous models
5. Process and save the embeddings

In [ ]:
print("🧬 Extracting embeddings from scCello...")

# Load pretrained scCello model
saved_model_path = f"{parent_model_dir}/scCello"
model = PrototypeContrastiveModel.from_pretrained(saved_model_path)
token_emb = model.state_dict()['embeddings.word_embeddings.weight']

print(f"🤖 scCello model loaded from: {saved_model_path}")
print(f"🔍 Embedding shape: {token_emb.shape} (e.g., [25427, 256])")
print(f"⚠️  Removing last token (CLS)")

In [ ]:
# Process scCello embeddings (removing CLS token)
gene_emb_df = pd.DataFrame(token_emb.numpy()[:-1, :])  # remove cls token
gene_emb_df["ENSG_ID"] = vocab.keys()
gene_emb_df["Symbol"] = gene_emb_df.apply(lambda x: gene_id_name.get(x["ENSG_ID"], None), axis=1)

# Merge with reference gene set
gene_emb_df = pd.merge(left=gene_set_df, right=gene_emb_df, on="Symbol", how="inner")
gene_emb_df.set_index("GeneID", inplace=True)
gene_emb_df.drop(["Symbol", "ENSG_ID"], axis=1, inplace=True)

# Save to CSV
output_file = f"./FRoGS/gene_embs/gene_vec_sccello_{gene_emb_df.shape[1]}.csv"
gene_emb_df.to_csv(output_file, header=None)

print(f"✅ scCello embeddings saved: {gene_emb_df.shape}")
print(f"📁 File: {output_file}")

## 7. Extract Gene Embeddings with scGPT

**scGPT** is a generative transformer pretrained for single-cell analysis. It requires specific configuration of parameters such as input bins, seed, and maximum sequence length.

### Procedure:
1. Configure model-specific parameters (bins, seed, max_seq_len)
2. Create an instance of the scGPT model and load the pretrained model
3. Extract embeddings from the encoder embedding layer
4. Process the scGPT-specific vocabulary
5. Save the processed embeddings

In [ ]:
print("🧬 Extracting embeddings from scGPT...")

# Configure scGPT-specific parameters
model_dir = f"{parent_model_dir}/scgpt/scGPT_human"
input_bins = 51        # Number of bins for expression discretization
seed = 7               # Seed for reproducibility  
n_hvg = 1200           # Number of highly variable genes
max_seq_len = n_hvg + 1  # Maximum sequence length (genes + special token)

print(f"⚙️  scGPT configuration:")
print(f"   📁 Model: {model_dir}")
print(f"   🔢 Input bins: {input_bins}")
print(f"   🌱 Seed: {seed}")
print(f"   📏 Max sequence length: {max_seq_len}")


In [ ]:
# Load and configure scGPT model
scgpt_model = scgpt_forward.scGPT_instance(saved_model_path=model_dir)
scgpt_model.create_configs(seed=seed, 
                          max_seq_len=max_seq_len, 
                          n_bins=input_bins)
scgpt_model.load_pretrained_model()

print(f"🤖 scGPT model loaded and configured")


In [ ]:
# Extract embeddings from the encoder of scGPT
token_emb = scgpt_model.model.state_dict()['encoder.embedding.weight']
print(f"🔍 Shape of scGPT embeddings: {token_emb.shape}")

# Process scGPT-specific vocabulary
vocab_list = scgpt_model.vocab.get_stoi().keys()
itos = {v: k for k, v in scgpt_model.vocab.get_stoi().items()}
sorted_dict = dict(sorted(itos.items()))

print(f"📚 scGPT vocabulary: {len(vocab_list)} tokens")
print(f"First 5 tokens: {list(sorted_dict.values())[:5]}")


In [ ]:
# Process and save scGPT embeddings
gene_emb_df = pd.DataFrame(token_emb.cpu().numpy())
gene_emb_df["Symbol"] = sorted_dict.values()

# Merge with reference gene set
gene_emb_df = pd.merge(left=gene_set_df, right=gene_emb_df, on="Symbol", how="inner")
gene_emb_df.set_index("GeneID", inplace=True)
gene_emb_df.drop(["Symbol"], axis=1, inplace=True)

# Save to CSV
output_file = f"./FRoGS/gene_embs/gene_vec_scgpt_{gene_emb_df.shape[1]}.csv"
gene_emb_df.to_csv(output_file, header=None)

print(f"✅ scGPT embeddings saved: {gene_emb_df.shape}")
print(f"📁 File: {output_file}")

## 8. Extract Gene Embeddings with UCE

**UCE (Universal Cell Embeddings)** uses ESM2 protein embeddings to represent genes. The embeddings are already precomputed and saved in a .pt file.

### Procedure:
1. Load precomputed embeddings from the .pt file
2. Transpose the tensor to obtain the correct structure
3. Use the indices as gene symbols
4. Process and save the embeddings

**⚠️ Important Note:** You must first run the script `UCE/eval_single_anndata.py` to download the model files.


In [ ]:
print("🧬 Extracting embeddings from UCE...")
print("⚠️  NOTE: Make sure you have run 'UCE/eval_single_anndata.py' first")

try:
    # Load precomputed UCE embeddings
    gene_emb = torch.load("./UCE/model_files/protein_embeddings/Homo_sapiens.GRCh38.gene_symbol_to_embedding_ESM2.pt")
    print(f"🔍 UCE embeddings loaded")
    print(f"📊 Data type: {type(gene_emb)}")
    print(f"🧬 Available genes: {len(gene_emb)}")
    
    # Show some example genes
    sample_genes = list(gene_emb.keys())[:5]
    print(f"Example genes: {sample_genes}")
    
except FileNotFoundError:
    print("❌ Error: UCE embeddings file not found")
    print("🔧 Please run: python UCE/eval_single_anndata.py first")


In [ ]:
# Process UCE embeddings
gene_emb_df = pd.DataFrame(gene_emb).T  # Transpose to have genes as rows
gene_emb_df["Symbol"] = gene_emb_df.index  # Use indices as gene symbols

print(f"🔍 Initial DataFrame shape: {gene_emb_df.shape}")
print(f"First 3 rows and 5 columns:")
print(gene_emb_df.iloc[:3, :5])

In [ ]:
# Merge with reference gene set and save
gene_emb_df = pd.merge(left=gene_set_df, right=gene_emb_df, on="Symbol", how="inner")
gene_emb_df.set_index("GeneID", inplace=True)
gene_emb_df.drop(["Symbol"], axis=1, inplace=True)

# Save to CSV
output_file = f"./FRoGS/gene_embs/gene_vec_uce_{gene_emb_df.shape[1]}.csv"
gene_emb_df.to_csv(output_file, header=None)

print(f"✅ UCE embeddings saved: {gene_emb_df.shape}")
print(f"📁 File: {output_file}")
print(f"🧬 Genes processed: {len(gene_emb_df)} of {len(gene_emb)} available in UCE")


## 🎉 Summary of Results

This notebook has successfully extracted gene embeddings from 6 different foundation models:

| Model         | Description                                      | Embedding Dimension | Output File                |
|---------------|--------------------------------------------------|---------------------|----------------------------|
| **xTrimoGene** | Foundation model for single-cell genomics        | Variable            | `gene_vec_xtrimogene_X.csv`|
| **Geneformer** | Transformer pretrained on transcriptomes         | Variable            | `gene_vec_geneformer_X.csv`|
| **LangCell**   | Combines cell biology + language modeling        | Variable            | `gene_vec_langcell_X.csv`  |
| **scCello**    | Contrastive learning for single-cell analysis    | Variable            | `gene_vec_sccello_X.csv`   |
| **scGPT**      | Generative transformer for single-cell analysis  | Variable            | `gene_vec_scgpt_X.csv`     |
| **UCE**        | Universal embeddings using ESM2                  | Variable            | `gene_vec_uce_X.csv`       |

### 📁 File Location
All embeddings have been saved in: `./FRoGS/gene_embs/`

### 🔄 Next Steps
These embeddings can be used for:
- 🧬 Gene function prediction
- 🕸️ Regulatory network analysis
- 📊 Comparative analysis between foundation models
- 🤖 Machine learning tasks in single-cell genomics

### 📝 Important Notes
- All embeddings are standardized using the same reference gene set
- The CSV files do not have headers (required format for FRoGS)
- Genes are indexed by GeneID for consistency across models
